# Learned Correction: Training & Evaluation

Train a small U-Net to predict the residual between the ideal and non-ideal
simulators, then use the corrected forward model to optimize phasors that
work under non-ideal conditions.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

from wavetank import (
    Tank, Actuator, build_propagator,
    steady_state_amplitudes, caustic_image,
    pack_complex, unpack_complex,
    cosine_loss, load_target_image,
    Stage, optimize_caustic,
    setup_from_target,
    NonIdealHyperparams, sample_nonideal_config,
    caustic_image_nonideal, sample_random_phasors,
    CorrectionUNet, apply_correction,
    generate_training_data, train_correction,
    make_corrected_loss,
)

print(f"JAX {jax.__version__} on {jax.default_backend()}")

## 1. Tank setup and non-ideal "reality"

We reuse the same tank geometry as the optimization notebook, then sample
a fixed non-ideal configuration that represents our "real" tank.

In [ ]:
# Tank geometry
tank = Tank(Lx=1.0, Ly=1.0, depth=0.12, damping=0.02)
nx, ny = 64, 64

# Actuators: 5 per side
acts = []
for i in range(5):
    t = (i + 1) / 6
    acts += [
        Actuator(x=0.0, y=t * tank.Ly),
        Actuator(x=tank.Lx, y=t * tank.Ly),
        Actuator(x=t * tank.Lx, y=0.0),
        Actuator(x=t * tank.Lx, y=tank.Ly),
    ]
n_act = len(acts)

prop = build_propagator(tank, acts, n_modes=12, nx=nx, ny=ny)
Omega = jnp.array([6.28, 9.42, 12.57])  # ~1, 1.5, 2 Hz
T_eval = 1.0

print(f"Propagator: {len(prop.omega)} modes, {prop.n_act} actuators, {len(Omega)} frequencies")
print(f"Parameters: {2 * prop.n_act * len(Omega)}")

# Sample a non-ideal configuration (this is our fixed "reality")
hyper = NonIdealHyperparams(
    damping_alpha=0.01,
    omega_sigma=0.005,
    coupling_sigma=0.02,
    distortion_k1_range=0.1,
    distortion_k2_range=0.01,
    distortion_center_sigma=0.02,
)
config = sample_nonideal_config(prop, hyper, jax.random.PRNGKey(42))
print(f"\nNon-ideal config sampled:")
print(f"  k1={config.k1:.4f}, k2={config.k2:.4f}")
print(f"  Damping range: {config.gamma.min():.4f} – {config.gamma.max():.4f}")
print(f"  Omega jitter std: {np.std(config.omega / prop.omega - 1):.4f}")

## 2. Visualize ideal vs non-ideal

Pick a few random phasor configurations and compare what the ideal simulator
produces vs what the non-ideal "reality" produces.

In [ ]:
n_freq = len(Omega)
sigma_render = 0.02

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row in range(3):
    key = jax.random.PRNGKey(row + 10)
    params = sample_random_phasors(key, prop.n_act, n_freq, scale=0.3)
    X, Y = unpack_complex(params, prop.n_act, n_freq)
    P = X + 1j * Y

    # Ideal
    a_id = steady_state_amplitudes(prop, P, Omega, T_eval)
    _, _, I_id = caustic_image(prop, a_id, sigma=sigma_render)

    # Non-ideal
    _, _, I_ni = caustic_image_nonideal(prop, config, P, Omega, T_eval, sigma=sigma_render)

    delta = I_ni - I_id
    vmax = max(float(jnp.max(I_id)), float(jnp.max(I_ni)))

    axes[row, 0].imshow(np.asarray(I_id).T, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    axes[row, 1].imshow(np.asarray(I_ni).T, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    im = axes[row, 2].imshow(np.asarray(delta).T, origin='lower', cmap='RdBu_r',
                              vmin=-0.5, vmax=0.5)

    if row == 0:
        axes[row, 0].set_title('Ideal')
        axes[row, 1].set_title('Non-ideal')
        axes[row, 2].set_title('Residual (NI - Ideal)')

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

## 3. Generate training data

Feed random phasors through both simulators to build a dataset of
(I_ideal, delta_I) pairs. The U-Net will learn to predict `delta_I` from `I_ideal`.

In [ ]:
%%time

N_TRAIN = 300  # increase to 500-1000 for better results

I_ideals, delta_Is = generate_training_data(
    prop, config, Omega, T_eval,
    n_samples=N_TRAIN,
    key=jax.random.PRNGKey(0),
    phasor_scale=0.5,
    sigma=sigma_render,
)

print(f"Training data: {I_ideals.shape}")
print(f"Mean |I_ideal|:  {float(jnp.mean(jnp.abs(I_ideals))):.4f}")
print(f"Mean |delta_I|:  {float(jnp.mean(jnp.abs(delta_Is))):.4f}")
print(f"Relative delta:  {float(jnp.mean(jnp.abs(delta_Is)) / jnp.mean(jnp.abs(I_ideals))):.1%}")

## 4. Train the correction U-Net

In [ ]:
model = CorrectionUNet(ch=16, key=jax.random.PRNGKey(0))

# Count parameters
import jax.tree_util
n_params = sum(x.size for x in jax.tree_util.tree_leaves(model))
print(f"U-Net parameters: {n_params:,}")

In [ ]:
%%time

model_trained, losses = train_correction(
    model, I_ideals, delta_Is,
    lr=1e-3,
    n_epochs=200,
    batch_size=16,
    key=jax.random.PRNGKey(1),
    verbose=True,
)

In [ ]:
plt.figure(figsize=(8, 3))
plt.semilogy(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Correction U-Net training')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss: {losses[0]:.6f} → {losses[-1]:.6f}  ({losses[-1]/losses[0]:.1%} of initial)")

## 5. Evaluate correction quality

Compare the U-Net's predicted correction against the actual non-ideal residual
on held-out phasor configurations (not seen during training).

In [ ]:
# Generate held-out test samples
I_test, delta_test = generate_training_data(
    prop, config, Omega, T_eval,
    n_samples=4, key=jax.random.PRNGKey(999),
    phasor_scale=0.5, sigma=sigma_render,
)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
cols = ['Ideal', 'Actual non-ideal', 'Corrected (ideal + UNet)', 'Residual error']

for row in range(4):
    I_id = I_test[row]
    delta_true = delta_test[row]
    I_ni_true = I_id + delta_true

    # Apply learned correction
    I_corrected = apply_correction(model_trained, I_id)

    vmax = max(float(jnp.max(I_id)), float(jnp.max(I_ni_true)))

    axes[row, 0].imshow(np.asarray(I_id).T, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    axes[row, 1].imshow(np.asarray(I_ni_true).T, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    axes[row, 2].imshow(np.asarray(I_corrected).T, origin='lower', cmap='inferno', vmin=0, vmax=vmax)

    err = I_corrected - I_ni_true
    axes[row, 3].imshow(np.asarray(err).T, origin='lower', cmap='RdBu_r', vmin=-0.3, vmax=0.3)

    # Cosine similarity: corrected vs true non-ideal
    cos_corr = float(1 - cosine_loss(I_corrected, I_ni_true))
    cos_uncorr = float(1 - cosine_loss(I_id, I_ni_true))
    axes[row, 0].set_ylabel(f'cos: {cos_uncorr:.3f}→{cos_corr:.3f}', fontsize=10)

    if row == 0:
        for c, title in enumerate(cols):
            axes[0, c].set_title(title)

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

## 6. Optimizing against the corrected model

The real test: optimize phasors to hit a target image, using the corrected
forward model. Compare three approaches:

1. **Ideal-only**: optimize against ideal simulator, evaluate on non-ideal
2. **Corrected**: optimize against ideal + learned correction, evaluate on non-ideal
3. **Oracle**: optimize directly against non-ideal simulator (best possible, but cheating)

In [ ]:
# Create a target: ring pattern
xs_grid = np.linspace(0, tank.Lx, nx)
ys_grid = np.linspace(0, tank.Ly, ny)
XX, YY = np.meshgrid(xs_grid, ys_grid, indexing='ij')
r = np.sqrt((XX - 0.5)**2 + (YY - 0.5)**2)
target = np.exp(-((r - 0.28)**2) / (2 * 0.08**2)).astype(np.float32)
target /= target.max()

plt.figure(figsize=(3, 3))
plt.imshow(target.T, origin='lower', cmap='inferno')
plt.title('Target')
plt.axis('off')
plt.show()

In [ ]:
%%time

# --- Approach 1: Optimize against ideal simulator ---
stages = (
    Stage(sigma=0.04, sigma_blur=0.04, iters=300),
    Stage(sigma=0.02, sigma_blur=0.02, iters=400),
    Stage(sigma=0.01, sigma_blur=0.01, iters=300),
)

params_ideal, _ = optimize_caustic(
    prop, target, np.asarray(Omega), T_eval,
    stages=stages, lr=5e-4, loss_type='cosine',
)
print("Ideal optimization done.")

In [ ]:
%%time

# --- Approach 2: Optimize against corrected model ---
import jaxopt

corrected_loss = make_corrected_loss(
    prop, model_trained, target, np.asarray(Omega), T_eval,
    sigma=0.02, lambda_energy=1e-5,
)

# Use the ideal-optimized phasors as warm start, refine with corrected model
solver = jaxopt.LBFGS(fun=corrected_loss, maxiter=200, history_size=20)
params_corrected, state = solver.run(jnp.asarray(params_ideal))
print(f"Corrected optimization: loss={float(state.value):.4f}, "
      f"iters={int(state.iter_num)}, grad_norm={float(state.error):.2e}")

In [ ]:
# --- Evaluate all three on the non-ideal "reality" ---
def eval_on_nonideal(params, label):
    """Run phasors through non-ideal simulator and measure cosine sim to target."""
    X, Y = unpack_complex(jnp.asarray(params), prop.n_act, n_freq)
    P = X + 1j * Y
    _, _, I_ni = caustic_image_nonideal(prop, config, P, Omega, T_eval, sigma=0.01)
    cos = float(1 - cosine_loss(I_ni, jnp.asarray(target)))
    print(f"  {label:20s}: cosine similarity = {cos:.4f}")
    return np.asarray(I_ni), cos

I_ni_ideal, cos_ideal = eval_on_nonideal(params_ideal, "Ideal-optimized")
I_ni_corrected, cos_corr = eval_on_nonideal(params_corrected, "Corrected-optimized")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
vmax = max(I_ni_ideal.max(), I_ni_corrected.max())

axes[0].imshow(target.T, origin='lower', cmap='inferno')
axes[0].set_title('Target')

axes[1].imshow(I_ni_ideal.T, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
axes[1].set_title(f'Ideal-opt on NI (cos={cos_ideal:.3f})')

axes[2].imshow(I_ni_corrected.T, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
axes[2].set_title(f'Corrected-opt on NI (cos={cos_corr:.3f})')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Non-ideal "reality" output: ideal vs corrected optimization', y=1.02)
plt.tight_layout()
plt.show()

improvement = cos_corr - cos_ideal
print(f"\nCosine similarity improvement: {improvement:+.4f}"
      f" ({cos_ideal:.4f} → {cos_corr:.4f})")